In [1]:
from model.tiny_yolo import TinyYolo  
from settings import Settings  
import os  
  
# ============================================================  
# STEP 1: Update Settings for 2-class training  
# ============================================================  
# Make sure settings.py has:  
# - class_names = ["Barcode", "QR"]  
# - train_dataset = "data/train_barcode_qr.tf_record"  
# - val_dataset = "data/val_barcode_qr.tf_record"  
  
print("Current settings:")  
print(f"Classes: {Settings.class_names}")  
print(f"Train dataset: {Settings.train['train_dataset']}")  
print(f"Val dataset: {Settings.train['val_dataset']}") 

Current settings:
Classes: ['Barcode', 'QR']
Train dataset: data/train_barcode_qr.tf_record
Val dataset: data/val_barcode_qr.tf_record


In [3]:
# ============================================================  
# STEP 2: Create model with 2 classes and load checkpoint  
# ============================================================  
# Create 2-class model  
tinyolo = TinyYolo(training=True, classes=2)  
model = tinyolo._gen_model()  
  
# Create temporary 1-class model to load checkpoint  
tinyolo_1class = TinyYolo(training=True, classes=1)  
model_1class = tinyolo_1class._gen_model()  
  
# Load epoch 99 checkpoint into 1-class model  
checkpoint_path = "checkpoints/yolov3_train_99.weights.h5"  
print(f"\nLoading weights from: {checkpoint_path}")  
model_1class.load_weights(checkpoint_path)  
  
# Transfer ONLY the backbone weights to 2-class model  
print("Transferring backbone weights...")  
model.get_layer('yolo_darknet').set_weights(  
    model_1class.get_layer('yolo_darknet').get_weights()  
)  
print("✓ Backbone weights transferred successfully")  
print("✓ Detection heads initialized with random weights for 2 classes")


Loading weights from: checkpoints/yolov3_train_99.weights.h5
Transferring backbone weights...
✓ Backbone weights transferred successfully
✓ Detection heads initialized with random weights for 2 classes


In [4]:
# ============================================================  
# STEP 3: Configure separate logs directory  
# ============================================================  
# Override the logs directory to keep separate from original training  
import datetime  
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")  
new_log_dir = f"logs/finetune_2class_{timestamp}/"  
os.makedirs(new_log_dir, exist_ok=True)  
  
print(f"\nNew logs will be saved to: {new_log_dir}")  
print("Original logs remain in: logs/") 


New logs will be saved to: logs/finetune_2class_20251017-135150/
Original logs remain in: logs/


In [5]:
# ============================================================  
# STEP 4: Start fine-tuning  
# ============================================================  
# Temporarily override the log directory  
original_log_dir = Settings.train["logs"]  
Settings.train["logs"] = new_log_dir  
  
print("\n" + "="*60)  
print("Starting fine-tuning with 2 classes (Barcode + QR)")  
print("="*60)  
  
tinyolo.train(skip_transfer_learning=True)  


Starting fine-tuning with 2 classes (Barcode + QR)
Epoch 1/100
     46/Unknown 393s 8s/step - loss: 1129.1588 - yolo_output_0_loss: 240.0645 - yolo_output_1_loss: 887.7045

2025-10-17 13:58:39.201221: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
/Users/glory/Downloads/Barcode-detection/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: saving model to checkpoints/yolov3_train_2class_1.weights.h5


2025-10-17 13:58:54.426578: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


46/46 ━━━━━━━━━━━━━━━━━━━━ 408s 9s/step - loss: 517.6951 - yolo_output_0_loss: 100.0827 - yolo_output_1_loss: 416.1791 - val_loss: 246.3618 - val_yolo_output_0_loss: 29.0076 - val_yolo_output_1_loss: 215.8688 - learning_rate: 0.0010
Epoch 2/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - loss: 56.7712 - yolo_output_0_loss: 8.0590 - yolo_output_1_loss: 47.2236

...

2025-10-17 18:31:27.977067: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Epoch 37: saving model to checkpoints/yolov3_train_2class_37.weights.h5


2025-10-17 18:31:44.811696: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


46/46 ━━━━━━━━━━━━━━━━━━━━ 407s 9s/step - loss: 4.3498 - yolo_output_0_loss: 0.8228 - yolo_output_1_loss: 2.0947 - val_loss: 5.7764 - val_yolo_output_0_loss: 0.6230 - val_yolo_output_1_loss: 3.7216 - learning_rate: 1.0000e-04
Epoch 38/100
26/46 ━━━━━━━━━━━━━━━━━━━━ 2:56 9s/step - loss: 4.5705 - yolo_output_0_loss: 0.7733 - yolo_output_1_loss: 2.3655

: 

In [ ]:
# Restore original log directory  
Settings.train["logs"] = original_log_dir  
  
print("\n" + "="*60)  
print("Fine-tuning complete!")  
print("="*60)  
print(f"View original training: tensorboard --logdir={original_log_dir}")  
print(f"View fine-tuning: tensorboard --logdir={new_log_dir}")  
print(f"Compare both: tensorboard --logdir=logs/")